# Llama 3 Inference Notebook

This notebook demonstrates how to use the Llama 3 model for text generation and chat completion.

## Features
- Text completion from prompts
- Chat completion with conversational formatting
- Multi-turn conversations
- Configurable generation parameters (temperature, top-p, max length)

## Setup for Kaggle

**Before running this notebook in Kaggle:**

### Step 1: Create a Kaggle Dataset

1. **Go to Kaggle Datasets:**
   - Visit https://www.kaggle.com/datasets
   - Click "New Dataset" button

2. **Upload your files from USB:**
   - **If you have safetensors files** (recommended for first upload):
     - Upload all 4 safetensors files:
       - `model-00001-of-00004.safetensors` (~4GB)
       - `model-00002-of-00004.safetensors` (~4GB)
       - `model-00003-of-00004.safetensors` (~4GB)
       - `model-00004-of-00004.safetensors` (~1.1GB)
     - Upload `config.json` (from Hugging Face model)
     - Upload `tokenizer.model`
   
   - **If you already have consolidated.00.pth:**
     - Upload `consolidated.00.pth` (~15GB)
     - Upload `params.json`
     - Upload `tokenizer.model`

3. **Name your dataset:**
   - Give it a name (e.g., "llama-3-8b-instruct")
   - Make it **Private** (recommended) or Public
   - Click "Create"

### Step 2: Add Dataset to Your Notebook

1. **In your Kaggle notebook:**
   - Click "Add data" button (top right)
   - Search for your dataset name
   - Click "Add" next to your dataset
   - The files will be available at `/kaggle/input/your-dataset-name/`

### Step 3: Update Configuration

- In the Configuration cell below, update:
  ```python
  dataset_name = "your-dataset-name"  # Change this!
  ```

### Step 4: Enable GPU

- Go to **Settings** → **Accelerator** → Select **"GPU T4 x2"** or higher
- The model requires GPU for inference

### Step 5: Upload Project Code

**Option A: Upload from USB (if code is on USB):**
- In Kaggle notebook, click "File" → "Upload"
- Upload the entire project folder (or zip it first)
- Extract to `/kaggle/working/` if needed

**Option B: Clone from GitHub (if code is in a repo):**
- Add a code cell at the top:
  ```python
  !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/
  ```

**Option C: Manual upload via Kaggle API:**
- Use Kaggle API to upload files programmatically


## Setup and Imports

First, we need to install dependencies (if needed) and set up the Python path to import the necessary modules.

**Note:** If you haven't uploaded the project code yet, choose one of these methods:

### Method 1: Git Clone (Recommended - Most Dynamic)
If your code is in a GitHub repo, run this cell to clone it:
```python
# Uncomment and update the URL if using git:
# !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/
```

### Method 2: Upload via Notebook
- Click "File" → "Upload" in the notebook
- Select individual files or folders
- Files will be in `/kaggle/working/`

### Method 3: Use Kaggle's Git Integration
- In notebook settings, enable "Git" integration
- Connect your GitHub repo


In [ ]:
# ============================================================================
# OPTION 1: Clone from GitHub (Easiest & Most Dynamic)
# ============================================================================
# If your code is in a GitHub repo, uncomment and run this:
# !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/

# ============================================================================
# OPTION 2: Install Dependencies (if needed)
# ============================================================================
# Uncomment if packages are missing:
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# %pip install -q tiktoken

# ============================================================================
# Setup Python Path
# ============================================================================
import sys
import os
from pathlib import Path

# Detect if running in Kaggle
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # In Kaggle: add /kaggle/working to path
    project_root = Path('/kaggle/working')
    sys.path.insert(0, str(project_root))
    print("Running in Kaggle environment")
    print(f"Project root: {project_root}")
    
    # Check if src directory exists
    if not (project_root / "src").exists():
        print("\n⚠️  WARNING: src/ directory not found!")
        print("\nTo fix this, choose one of these options:")
        print("\n1. Git Clone (Recommended):")
        print("   !git clone https://github.com/your-username/llama-3-from-scratch.git /kaggle/working/")
        print("\n2. Upload Files:")
        print("   - Click 'File' → 'Upload' in the notebook")
        print("   - Upload the 'src' folder and other project files")
        print("\n3. Use Kaggle's Git Integration:")
        print("   - Go to Settings → Enable 'Git'")
        print("   - Connect your GitHub repo")
        raise FileNotFoundError("Project code not found. Please upload or clone the code first.")
    else:
        print("✓ Project code found")
else:
    # Local: add parent directory to path
    project_root = Path().resolve().parent
    sys.path.insert(0, str(project_root))
    print("Running in local environment")
    print(f"Project root: {project_root}")

# Import the Llama inference class
try:
    from src.inference import Llama
    print("✓ Successfully imported Llama class")
except ImportError as e:
    print(f"❌ Failed to import Llama: {e}")
    print("\nMake sure the project code is in the correct location:")
    if IS_KAGGLE:
        print("  - Upload to /kaggle/working/")
        print("  - Or clone from GitHub")
    else:
        print("  - Should be in parent directory")
    raise


## Configuration

Configure the paths to your model files. You can provide either:
- **Safetensors files** (`model-00001-of-00004.safetensors`, etc.) - will be converted automatically
- **Consolidated checkpoint** (`consolidated.00.pth`) - ready to use

Also configure generation parameters.


In [ ]:
# Detect if running in Kaggle
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # Kaggle paths: model should be in /kaggle/input/your-dataset-name/
    # Update 'your-dataset-name' to match your actual dataset name
    dataset_name = "llama3"  # CHANGE THIS to your dataset name
    model_dir = f"/kaggle/input/{dataset_name}/"  # Directory containing model files
    # Output directory for converted files (if needed)
    output_dir = "/kaggle/working/checkpoints"  # Where to save consolidated.00.pth if converting
    tokenizer_path = f"/kaggle/input/{dataset_name}/tokenizer.model"  # Path to tokenizer.model file
    print(f"Using Kaggle paths:")
    print(f"  Model directory: {model_dir}")
    print(f"  Output directory: {output_dir}")
    print(f"  Tokenizer: {tokenizer_path}")
else:
    # Local paths
    model_dir = "./checkpoints"  # Directory containing model files (safetensors or consolidated)
    output_dir = "./checkpoints"  # Where to save consolidated.00.pth if converting
    tokenizer_path = "./checkpoints/tokenizer.model"  # Path to tokenizer.model file
    print(f"Using local paths:")
    print(f"  Model directory: {model_dir}")
    print(f"  Output directory: {output_dir}")
    print(f"  Tokenizer: {tokenizer_path}")

# Generation parameters
temperature = 0.6  # Sampling temperature (0.0 = deterministic, higher = more random)
top_p = 0.9       # Top-p (nucleus) sampling parameter
max_seq_len = 128 # Maximum sequence length for the model
max_gen_len = 64  # Maximum number of tokens to generate
max_batch_size = 4  # Maximum batch size for processing multiple prompts


## Convert Safetensors to Consolidated Format (if needed)

If you have safetensors files instead of consolidated.00.pth, this cell will convert them automatically.


In [ ]:
import os
import json
from pathlib import Path
import torch

# Check if consolidated.00.pth already exists
model_path = Path(model_dir)
consolidated_path = model_path / "consolidated.00.pth"
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)
final_consolidated = output_path / "consolidated.00.pth"

# Check if we already have consolidated.00.pth
if consolidated_path.exists():
    print(f"✓ Found consolidated.00.pth: {consolidated_path}")
    print(f"  Size: {consolidated_path.stat().st_size / 1024**3:.2f} GB")
    ckpt_dir = str(model_path)  # Use existing consolidated file
    print("\n✓ Ready to use! Skipping conversion.")
elif final_consolidated.exists():
    print(f"✓ Found consolidated.00.pth in output directory: {final_consolidated}")
    print(f"  Size: {final_consolidated.stat().st_size / 1024**3:.2f} GB")
    ckpt_dir = str(output_path)  # Use converted file
    print("\n✓ Ready to use! Skipping conversion.")
else:
    # Check for safetensors files
    print("Checking for safetensors files...")
    safetensors_files = sorted(model_path.glob("model-*.safetensors"))
    
    if not safetensors_files:
        print("❌ No safetensors files found!")
        print(f"   Looked in: {model_path}")
        print("\nPlease ensure you have either:")
        print("  1. consolidated.00.pth file, OR")
        print("  2. model-00001-of-00004.safetensors, model-00002-of-00004.safetensors, etc.")
        raise FileNotFoundError("No model files found!")
    
    print(f"✓ Found {len(safetensors_files)} safetensors file(s)")
    for st_file in safetensors_files:
        size_gb = st_file.stat().st_size / 1024**3
        print(f"  - {st_file.name} ({size_gb:.2f} GB)")
    
    # Check for config.json
    config_path = model_path / "config.json"
    if not config_path.exists():
        print(f"\n⚠️  Warning: config.json not found in {model_path}")
        print("   You may need to download it from Hugging Face")
    
    # Convert safetensors to consolidated format
    print(f"\n🔄 Converting safetensors to consolidated.00.pth...")
    print("   This may take a few minutes...")
    
    try:
        from safetensors.torch import load_file
    except ImportError:
        print("Installing safetensors...")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "safetensors"])
        from safetensors.torch import load_file
    
    # Load all safetensors files
    state_dict = {}
    for st_file in safetensors_files:
        print(f"   Loading {st_file.name}...")
        tensors = load_file(str(st_file))
        state_dict.update(tensors)
    
    print(f"   ✓ Loaded {len(state_dict)} weight tensors")
    
    # Load config.json if available
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)
        
        # Create params.json
        params = {
            "dim": config["hidden_size"],
            "n_layers": config["num_hidden_layers"],
            "n_heads": config["num_attention_heads"],
            "n_kv_heads": config.get("num_key_value_heads", config["num_attention_heads"]),
            "vocab_size": config["vocab_size"],
            "multiple_of": config.get("multiple_of", 256),
            "ffn_dim_multiplier": config.get("ffn_dim_multiplier", None),
            "norm_eps": config["rms_norm_eps"],
            "rope_theta": config.get("rope_theta", 500000.0),
        }
        
        params_path = output_path / "params.json"
        with open(params_path, "w") as f:
            json.dump(params, f, indent=2)
        print(f"   ✓ Created params.json")
    else:
        print("   ⚠️  No config.json found - you'll need params.json separately")
    
    # Convert weight names from Hugging Face format to Meta format
    print("   Converting weight names...")
    meta_state_dict = {}
    
    for key, value in state_dict.items():
        # Remove 'model.' prefix if present
        new_key = key.replace("model.", "")
        
        # Convert layer names
        new_key = new_key.replace("self_attn.q_proj", "attention.wq")
        new_key = new_key.replace("self_attn.k_proj", "attention.wk")
        new_key = new_key.replace("self_attn.v_proj", "attention.wv")
        new_key = new_key.replace("self_attn.o_proj", "attention.wo")
        
        # Convert feedforward names
        new_key = new_key.replace("mlp.gate_proj", "feed_forward.w1")
        new_key = new_key.replace("mlp.up_proj", "feed_forward.w3")
        new_key = new_key.replace("mlp.down_proj", "feed_forward.w2")
        
        # Convert normalization names
        new_key = new_key.replace("input_layernorm", "attention_norm")
        new_key = new_key.replace("post_attention_layernorm", "ffn_norm")
        
        # Embeddings
        new_key = new_key.replace("embed_tokens", "tok_embeddings")
        
        # Output layer
        new_key = new_key.replace("lm_head", "output")
        
        meta_state_dict[new_key] = value
    
    print(f"   ✓ Converted {len(meta_state_dict)} weight tensors")
    
    # Save consolidated checkpoint
    print(f"   Saving consolidated.00.pth to {final_consolidated}...")
    torch.save(meta_state_dict, final_consolidated)
    size_gb = final_consolidated.stat().st_size / 1024**3
    print(f"   ✓ Saved consolidated.00.pth ({size_gb:.2f} GB)")
    
    ckpt_dir = str(output_path)  # Use converted file location
    print("\n✓ Conversion complete! Ready to load model.")


## Verify Files

Quick check that all required files are ready.


In [ ]:
# Final verification
ckpt_path = Path(ckpt_dir)
required_files = {
    "consolidated.00.pth": ckpt_path / "consolidated.00.pth",
    "params.json": ckpt_path / "params.json",
    "tokenizer.model": Path(tokenizer_path),
}

print("Final file check:")
all_found = True
for name, filepath in required_files.items():
    exists = filepath.exists()
    status = "✓" if exists else "✗"
    if exists:
        size = f" ({filepath.stat().st_size / 1024**3:.2f} GB)" if name.endswith('.pth') else ""
        print(f"  {status} {name}: {filepath}{size}")
    else:
        print(f"  {status} {name}: {filepath} - NOT FOUND")
        all_found = False

if not all_found:
    raise FileNotFoundError("Missing required files! Check paths above.")
else:
    print("\n✓ All files ready!")


## Load Model

Load the pre-trained Llama 3 model. This may take a few minutes as it loads ~8B parameters into GPU memory.


In [ ]:
print("Loading model...")
print("This may take a few minutes as it loads ~8B parameters into GPU memory...\n")

generator = Llama.build(
    ckpt_dir=ckpt_dir,
    tokenizer_path=tokenizer_path,
    max_seq_len=max_seq_len,
    max_batch_size=max_batch_size,
    model_parallel_size=1  # Set to 1 for single GPU, increase for multi-GPU
)
print("✓ Model loaded successfully!\n")


## Example 1: Text Completion

Generate text completions from simple prompts. The model will continue the given text.


In [ ]:
prompts = [
    "I believe the meaning of life is",
    "Simply put, the theory of relativity states that ",
]

# Generate completions for all prompts
results = generator.text_completion(
    prompts,
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)

# Print results
for prompt, result in zip(prompts, results):
    print(f"Prompt: {prompt}")
    print(f"Completion: {result['generation']}")
    print("\n" + "=" * 50 + "\n")


## Example 2: Chat Completion

Generate responses in a conversational format with system and user messages.


In [ ]:
# Define a conversation dialog
# Each message has a role (system, user, or assistant) and content
dialogs = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ]
]

# Generate assistant response
chat_results = generator.chat_completion(
    dialogs,
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)

# Print chat results
for dialog, result in zip(dialogs, chat_results):
    print("Dialog:")
    for message in dialog:
        print(f"  {message['role']}: {message['content']}")
    print(f"\nAssistant: {result['generation']['content']}")
    print("\n" + "=" * 50 + "\n")


## Example 3: Multi-turn Conversation

Demonstrate a multi-turn conversation where the model maintains context across multiple exchanges.


In [ ]:
# Start a conversation
conversation = [
    {"role": "user", "content": "Explain quantum computing in simple terms."},
]

# Generate first response
response1 = generator.chat_completion(
    [conversation],
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)[0]

# Add assistant response to conversation
conversation.append(response1["generation"])

# Add follow-up question
conversation.append({"role": "user", "content": "How does it differ from classical computing?"})

# Generate second response
response2 = generator.chat_completion(
    [conversation],
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)[0]

# Print the full conversation
print("Full conversation:")
for message in conversation:
    print(f"  {message['role']}: {message['content']}")
print(f"  {response2['generation']['role']}: {response2['generation']['content']}")


## Custom Prompts

Try your own prompts! Modify the cell below to test different inputs.


In [ ]:
# Custom text completion
custom_prompt = "The future of artificial intelligence will"

result = generator.text_completion(
    [custom_prompt],
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
)[0]

print(f"Prompt: {custom_prompt}")
print(f"Completion: {result['generation']}")


## Adjusting Generation Parameters

You can experiment with different generation parameters to control the output:

- **temperature**: Lower values (0.0-0.5) = more deterministic, focused outputs. Higher values (0.7-1.5) = more creative, diverse outputs.
- **top_p**: Nucleus sampling threshold. Lower values (0.5-0.8) = more focused on high-probability tokens. Higher values (0.9-1.0) = broader sampling.
- **max_gen_len**: Maximum number of tokens to generate. Increase for longer outputs.


In [ ]:
# Example with different parameters
test_prompt = "Write a short story about a robot learning to paint."

# More creative (higher temperature)
creative_result = generator.text_completion(
    [test_prompt],
    max_gen_len=100,
    temperature=0.9,  # Higher temperature for more creativity
    top_p=0.95,
)[0]

print("Creative output (temperature=0.9):")
print(creative_result['generation'])
print("\n" + "=" * 50 + "\n")

# More focused (lower temperature)
focused_result = generator.text_completion(
    [test_prompt],
    max_gen_len=100,
    temperature=0.3,  # Lower temperature for more focused output
    top_p=0.8,
)[0]

print("Focused output (temperature=0.3):")
print(focused_result['generation'])
